# 4주차 실습 — 헬스케어 데이터관리
목표: 걸음수·수면·활동 데이터의 품질을 점검하고 규칙 기반 알림과 Isolation Forest 이상탐지를 비교합니다.

> 교육용 분석이며 의료 진단이 아닙니다.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest


## 1. 데이터 불러오기
`week04_healthcare_steps_sleep_practice.csv`를 Colab에 업로드한 뒤 실행합니다.


In [ ]:
df = pd.read_csv('week04_healthcare_steps_sleep_practice.csv')
df.head()


## 2. 데이터 품질 점검


In [ ]:
print(df.shape)
print(df.dtypes)
display(df.isna().sum())
display(df.describe())


## 3. 규칙 기반 알림
교육용 예시: steps < 1000, sleep_hours < 4 또는 > 12, calories <= 0, very_active_minutes > 300


In [ ]:
def rule_alert(row):
    alerts = []
    if pd.notna(row['steps']) and row['steps'] < 1000: alerts.append('low_steps')
    if pd.notna(row['sleep_hours']) and row['sleep_hours'] < 4: alerts.append('short_sleep')
    if pd.notna(row['sleep_hours']) and row['sleep_hours'] > 12: alerts.append('long_sleep')
    if pd.notna(row['calories']) and row['calories'] <= 0: alerts.append('invalid_calories')
    if pd.notna(row['very_active_minutes']) and row['very_active_minutes'] > 300: alerts.append('active_minutes_outlier')
    return ','.join(alerts) if alerts else 'normal'
df['rule_alert'] = df.apply(rule_alert, axis=1)
df[df['rule_alert'] != 'normal']


## 4. Isolation Forest 이상탐지


In [ ]:
features = ['steps','sleep_hours','calories','sedentary_minutes','very_active_minutes']
X = df[features].copy().fillna(df[features].median(numeric_only=True))
model = IsolationForest(contamination=0.12, random_state=42)
pred = model.fit_predict(X)
df['ai_anomaly'] = np.where(pred == -1, 'anomaly', 'normal')
df[df['ai_anomaly'] == 'anomaly'][['date'] + features + ['rule_alert','ai_anomaly']]


## 5. 결과 비교


In [ ]:
df['rule_flag'] = np.where(df['rule_alert'] == 'normal', 'normal', 'anomaly')
display(pd.crosstab(df['rule_flag'], df['ai_anomaly'], margins=True))
display(df[df['rule_flag'] != df['ai_anomaly']][['date'] + features + ['rule_alert','rule_flag','ai_anomaly']])


## 6. 해석 질문
1. 규칙 기반만 이상으로 판단한 사례는?
2. AI만 이상으로 판단한 사례는?
3. 두 방법의 장단점은?
4. 실제 헬스케어 서비스에서 자동경고 전에 어떤 검증이 필요한가?
